# NDWI Analysis: Identifying Actively Irrigated Center Pivot Systems

## Purpose
Classify center pivot irrigation systems (CPIS) in arid Sub-Saharan Africa as actively
irrigated or dormant/abandoned using the Normalized Difference Water Index (NDWI)
derived from Sentinel-2 satellite imagery.

## Narrative context
This notebook is **Step 2** in the water source attribution arc:

1. *CPIS expansion* — Notebooks in `Code/1_analyze_data` established that CPIS expanded
   191% in SSA between 2000 and 2021.
2. **Which systems are actually active? ← This notebook**
   Chen et al. (2023) mapped every CPIS-shaped polygon, but some are fallow or
   abandoned. NDWI separates actively irrigated systems from inactive ones, providing
   a cleaned denominator for all downstream water-source analyses. Analyzing water
   source relationships for abandoned systems would conflate two distinct phenomena.
3. *Dam accessibility* — `6_dem_flow_analysis.ipynb` refines distance-to-dam analysis
   using elevation, establishing which active CPIS could plausibly use surface water.
4. *Groundwater* — `7_groundwater_gp_regression.ipynb` tests whether active CPIS
   correlate with groundwater availability.
5. *Anomalies* — `8_anomaly_detection.ipynb` isolates systems unexplained by either source.

NDWI = (Green − NIR) / (Green + NIR)
Positive values indicate open water or actively irrigated crops; negative values
indicate bare soil or dry vegetation.

## Inputs
| Dataset | Config key | Notes |
|---------|-----------|-------|
| Sentinel-2 L2A (B03 Green, B8A NIR) | `Sentinel2_dir_path` | **Must be downloaded — see Data Acquisition below** |
| CPIS polygons (arid SSA, combined) | `SSA_Combined_CPIS_All_shp_path` | From `0_process_data` |
| Africa boundaries | `Africa_boundaries_shp_path` | For visualization |
| Arid SSA regions | `SSA_All_by_Country_shp_path` | For visualization |

## Outputs
| File | Config key | Description |
|------|-----------|-------------|
| `CPIS_NDWI.tif` | `NDWI_output_tif_path` | NDWI raster over analysis region |
| `Active_CPIS.shp` | `Active_CPIS_shp_path` | CPIS with mean NDWI > 0 |
| `Inactive_CPIS.shp` | `Inactive_CPIS_shp_path` | CPIS with mean NDWI ≤ 0 |
| `CPIS_NDWI_Stats.csv` | `CPIS_NDWI_stats_csv_path` | Per-polygon NDWI statistics |

## Data Acquisition: Obtaining Sentinel-2 Imagery

Sentinel-2 Level-2A surface reflectance imagery is required but not included in the
repository. Two acquisition options:

### Option A: Google Earth Engine (recommended)
The GEE script below exports cloud-filtered median-composited Sentinel-2 bands
(B03 = Green, B8A = NIR) for the arid SSA region. Run at https://code.earthengine.google.com

```javascript
// ============================================================
// GEE Export: Sentinel-2 NDWI bands for arid SSA
// ============================================================
var aridSSA = ee.FeatureCollection('users/YOUR_USERNAME/SSA_All_Arid_by_Country');
// Upload SSA_All_Arid_by_Country.shp to your GEE assets first

var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(aridSSA)
  .filterDate('2021-01-01', '2021-12-31')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
  .select(['B3', 'B8A'])
  .median()
  .clip(aridSSA);

Export.image.toDrive({
  image: s2.select('B3').rename('B03_Green'),
  description: 'Sentinel2_B03_Green_AridSSA_2021',
  folder: 'Africa_Irrigation',
  region: aridSSA.geometry().bounds(),
  scale: 60,
  crs: 'EPSG:4326',
  maxPixels: 1e13
});
Export.image.toDrive({
  image: s2.select('B8A').rename('B8A_NIR'),
  description: 'Sentinel2_B8A_NIR_AridSSA_2021',
  folder: 'Africa_Irrigation',
  region: aridSSA.geometry().bounds(),
  scale: 60,
  crs: 'EPSG:4326',
  maxPixels: 1e13
});
```

### Option B: Copernicus Open Access Hub
1. Register at https://browser.dataspace.copernicus.eu/
2. Search for Sentinel-2 L2A tiles covering arid SSA
3. Download bands B03 and B8A; mosaic if needed (`gdal_merge.py`)

**Place downloaded files at:**
```
Data/Raw/Sentinel2/B03_Green.tif
Data/Raw/Sentinel2/B8A_NIR.tif
```

In [1]:
# --- Import Required Libraries and Utilities ---
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import mapping

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Code.utils.utility import load_config, resolve_path

config = load_config()

## Check Data Availability

In [2]:
# --- Check for Sentinel-2 input files ---
sentinel2_dir = resolve_path(config['Sentinel2_dir_path'])
b03_path = os.path.join(sentinel2_dir, 'B03_Green.tif')
b8a_path = os.path.join(sentinel2_dir, 'B8A_NIR.tif')

DATA_AVAILABLE = os.path.isfile(b03_path) and os.path.isfile(b8a_path)

if not DATA_AVAILABLE:
    print("WARNING: Sentinel-2 data not found.")
    print(f"  Expected: {b03_path}")
    print(f"  Expected: {b8a_path}")
    print("See the 'Data Acquisition' section above for download instructions.")
    print("Analysis cells below will skip until data is available.")
else:
    print("Sentinel-2 data found. Proceeding with NDWI analysis.")

  Expected: /home/waves/data/Africa_Irrigation/Data/Raw/Sentinel2/B03_Green.tif
  Expected: /home/waves/data/Africa_Irrigation/Data/Raw/Sentinel2/B8A_NIR.tif
See the 'Data Acquisition' section above for download instructions.
Analysis cells below will skip until data is available.


## Compute NDWI from Sentinel-2 Bands

NDWI = (Green − NIR) / (Green + NIR)

Values range from −1 to +1:
- **Positive (> 0):** Open water or actively irrigated vegetation
- **Near zero (−0.1 to 0):** Sparse or stressed vegetation
- **Negative (< −0.1):** Bare soil, dry vegetation, or built surfaces

For CPIS activity classification, NDWI > 0 is used as the threshold for active
irrigation. This can be tuned based on regional crop calendars and validation data.

In [3]:
# --- Load Sentinel-2 bands and compute NDWI ---
if DATA_AVAILABLE:
    with rasterio.open(b03_path) as src_green:
        green = src_green.read(1).astype('float32')
        profile = src_green.profile.copy()
        crs_s2 = src_green.crs
        nodata_val = src_green.nodata if src_green.nodata is not None else 0

    with rasterio.open(b8a_path) as src_nir:
        nir = src_nir.read(1).astype('float32')

    valid = (green != nodata_val) & (nir != nodata_val) & ((green + nir) > 0)
    ndwi = np.full_like(green, np.nan)
    ndwi[valid] = (green[valid] - nir[valid]) / (green[valid] + nir[valid])

    print(f"Band shape: {green.shape}")
    print(f"CRS: {crs_s2}")
    print(f"Valid pixels: {valid.sum():,}")
    print(f"NDWI range: [{np.nanmin(ndwi):.3f}, {np.nanmax(ndwi):.3f}]")
    print(f"Mean NDWI: {np.nanmean(ndwi):.3f}")

    # Write NDWI raster
    ndwi_out_path = resolve_path(config['NDWI_output_tif_path'])
    os.makedirs(os.path.dirname(ndwi_out_path), exist_ok=True)
    profile.update(dtype='float32', count=1, nodata=np.nan)
    with rasterio.open(ndwi_out_path, 'w', **profile) as dst:
        dst.write(ndwi[np.newaxis, ...])
    print(f"\nNDWI raster saved: {ndwi_out_path}")
else:
    print("[Skipped] Sentinel-2 data required. See Data Acquisition section.")

[Skipped] Sentinel-2 data required. See Data Acquisition section.


## Load CPIS Polygons

We use the combined (2000 + 2021) CPIS dataset for arid SSA. Each polygon represents
one center pivot system observed in either year.

In [4]:
# --- Load and reproject CPIS polygons ---
cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))

if DATA_AVAILABLE:
    # Read CRS directly from the Sentinel-2 file so this cell is self-contained
    # and does not depend on crs_s2 being defined in the ndwi-compute cell above.
    with rasterio.open(b03_path) as _src:
        cpis = cpis.to_crs(_src.crs)

print(f"CPIS polygons loaded: {len(cpis)}")
print(f"CRS: {cpis.crs}")
print(f"Columns: {list(cpis.columns)}")

CPIS polygons loaded: 29493
CRS: EPSG:3857
Columns: ['ID', 'Country', 'Country Co', 'Year', 'geometry']


## Extract NDWI Statistics per CPIS Polygon

For each CPIS polygon we extract all Sentinel-2 pixels within its boundary and
compute summary statistics. Mean NDWI drives the active/inactive classification;
standard deviation captures within-pivot variability.

In [5]:
# --- Extract per-polygon NDWI statistics ---
if DATA_AVAILABLE:
    ndwi_records = []
    ndwi_tif_path = resolve_path(config['NDWI_output_tif_path'])

    with rasterio.open(ndwi_tif_path) as src:
        for idx, row in cpis.iterrows():
            try:
                geom = [mapping(row.geometry)]
                out_image, _ = rasterio.mask.mask(
                    src, geom, crop=True, nodata=np.nan, all_touched=True
                )
                vals = out_image[0].flatten()
                vals = vals[~np.isnan(vals)]
                if len(vals) > 0:
                    ndwi_records.append({
                        'cpis_idx': idx,
                        'ndwi_mean': float(np.mean(vals)),
                        'ndwi_median': float(np.median(vals)),
                        'ndwi_std': float(np.std(vals)),
                        'n_pixels': len(vals)
                    })
                else:
                    ndwi_records.append(
                        {'cpis_idx': idx, 'ndwi_mean': np.nan,
                         'ndwi_median': np.nan, 'ndwi_std': np.nan, 'n_pixels': 0}
                    )
            except Exception:
                ndwi_records.append(
                    {'cpis_idx': idx, 'ndwi_mean': np.nan,
                     'ndwi_median': np.nan, 'ndwi_std': np.nan, 'n_pixels': 0}
                )

    ndwi_df = pd.DataFrame(ndwi_records).set_index('cpis_idx')
    cpis = cpis.join(ndwi_df)

    valid_count = cpis['ndwi_mean'].notna().sum()
    print(f"NDWI extracted for {valid_count:,} / {len(cpis):,} CPIS polygons")
    print(cpis[['ndwi_mean', 'ndwi_std', 'n_pixels']].describe().round(3))
else:
    print("[Skipped] Requires NDWI raster from previous cell.")

[Skipped] Requires NDWI raster from previous cell.


## Classify CPIS as Active or Inactive

**Threshold:** mean NDWI > 0 → actively irrigated

An NDWI > 0 within a CPIS polygon indicates more green/water reflectance than
bare-soil reflectance, consistent with active crop growth. Studies in dryland
agriculture commonly use thresholds between 0.0 and 0.1. Adjust `NDWI_THRESHOLD`
and compare with field validation data for your study region.

In [6]:
# --- Classify and save active/inactive CPIS ---
if DATA_AVAILABLE:
    NDWI_THRESHOLD = 0.0

    cpis['active'] = cpis['ndwi_mean'] > NDWI_THRESHOLD

    active_cpis = cpis[cpis['active'] == True].copy()
    inactive_cpis = cpis[cpis['active'] == False].copy()

    n_total = len(cpis)
    n_active = len(active_cpis)
    n_inactive = len(inactive_cpis)
    n_unknown = cpis['ndwi_mean'].isna().sum()

    print(f"Total CPIS: {n_total}")
    print(f"  Active   (NDWI > {NDWI_THRESHOLD}):  {n_active:5d} ({100*n_active/n_total:.1f}%)")
    print(f"  Inactive (NDWI <= {NDWI_THRESHOLD}): {n_inactive:5d} ({100*n_inactive/n_total:.1f}%)")
    print(f"  No data:                {n_unknown:5d} ({100*n_unknown/n_total:.1f}%)")

    active_path = resolve_path(config['Active_CPIS_shp_path'])
    inactive_path = resolve_path(config['Inactive_CPIS_shp_path'])
    stats_path = resolve_path(config['CPIS_NDWI_stats_csv_path'])

    os.makedirs(os.path.dirname(active_path), exist_ok=True)
    os.makedirs(os.path.dirname(inactive_path), exist_ok=True)

    active_cpis.to_file(active_path)
    inactive_cpis.to_file(inactive_path)
    ndwi_df.to_csv(stats_path)

    print(f"\nSaved active CPIS:   {active_path}")
    print(f"Saved inactive CPIS: {inactive_path}")
    print(f"Saved NDWI stats:    {stats_path}")
else:
    print("[Skipped] Requires NDWI extraction from previous cell.")

[Skipped] Requires NDWI extraction from previous cell.


## Visualize NDWI and CPIS Activity

Two panels:
- **Left:** NDWI raster with CPIS outlines
- **Right:** Active (green) vs inactive (red) CPIS on SSA boundaries

In [7]:
# --- Visualization ---
if DATA_AVAILABLE:
    from rasterio.plot import show as rshow

    af_bnds = gpd.read_file(resolve_path(config['Africa_boundaries_shp_path']))
    af_bnds['geometry'] = af_bnds['geometry'].simplify(0.01)
    arid_ssa = gpd.read_file(resolve_path(config['SSA_All_by_Country_shp_path']))
    arid_ssa['geometry'] = arid_ssa['geometry'].simplify(0.01)

    fig, axes = plt.subplots(1, 2, figsize=(28, 14))
    plt.subplots_adjust(wspace=0.05, left=0.05, right=0.95, top=0.9, bottom=0.05)

    # Panel 1: NDWI raster
    ax = axes[0]
    norm_ndwi = TwoSlopeNorm(vmin=-0.5, vcenter=0, vmax=0.5)
    with rasterio.open(resolve_path(config['NDWI_output_tif_path'])) as src:
        rshow(src, ax=ax, cmap='RdYlGn', norm=norm_ndwi, alpha=0.85)
    cpis.boundary.plot(ax=ax, color='black', linewidth=0.3, alpha=0.6)
    arid_ssa.boundary.plot(ax=ax, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title('NDWI with CPIS Outlines', fontsize=22)
    ax.set_axis_off()

    # Panel 2: Active vs inactive
    ax2 = axes[1]
    non_arid = af_bnds.overlay(arid_ssa, how='difference', keep_geom_type=False)
    non_arid.plot(ax=ax2, color='lightgray', hatch='//', edgecolor='black',
                  linewidth=0.5, alpha=0.6)
    arid_ssa.boundary.plot(ax=ax2, color='black', linewidth=1.0, linestyle='--')
    inactive_cpis.plot(ax=ax2, color='#d73027', markersize=4, alpha=0.7,
                       label=f'Inactive ({len(inactive_cpis):,})')
    active_cpis.plot(ax=ax2, color='#1a9850', markersize=4, alpha=0.7,
                     label=f'Active ({len(active_cpis):,})')
    ax2.set_title(f'Active vs Inactive CPIS (NDWI threshold = {NDWI_THRESHOLD})',
                  fontsize=22)
    ax2.set_axis_off()
    ax2.legend(fontsize=16, loc='lower left', framealpha=0.9)

    fig.suptitle('CPIS Activity Classification from Sentinel-2 NDWI',
                 fontsize=28, fontweight='bold', y=0.95)
    plt.show()

    print("\nNote: Active CPIS (green) have mean NDWI > 0, indicating active irrigation.")
    print("Inactive CPIS (red) may be fallow, abandoned, or imaged outside growing season.")
else:
    print("[PLACEHOLDER] Visualization requires Sentinel-2 data.")
    print("Download data following the instructions in the 'Data Acquisition' section above.")

[PLACEHOLDER] Visualization requires Sentinel-2 data.
Download data following the instructions in the 'Data Acquisition' section above.
